# 4 · Rules that take arguments

**The problem.** Many rules differ only in a value: "is named Ann", "is named Bob". Copying the rule once per value
copies its mistakes too. A predicate can take **parameters**, and be **applied** where it's used, by reference: the
same predicate object, written once.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Constraints, Predicates, Queries, Validators } from "@mbse/patterns";
import { JSON as SchemaJSON } from "@mbse/schemas/Framework";
import { Contact, book, listed } from "./toolkit.ts";

const store = book();
listed(store, ...([["Ann", 34n], ["Bob", 41n], ["Kid", 9n]] as const).map(([n, a]) => store.Contact().name(n).age(a).create()));

## Step 1: a parameter

`.parameters(...)` takes property specs, as an object schema's `.properties(...)` does. The rule's free names are the
symbols and the parameters:

In [2]:
const person = E.variable("person");
const HasName = new Predicates.OfPredicate.Builder().name("HasName").symbols({ person: Contact })
  .parameters((x) => x.name("name"))
  .requires(person.name.eq(E.variable("name"))).create();
console.log([...HasName.symbols.keys()], [...HasName.parameters.keys()]);

[ "person" ] [ "name" ]


## Step 2: applying it

`HasName.call(who, "Ann")` applies the predicate, binding its symbol, then its parameter (in Python, you call it
directly). A builder gives the variables it declares, so `named.c` is the variable `c`:

In [3]:
const named = new Predicates.OfPredicate.Builder().name("IsAnn").symbols({ c: Contact });
const IsAnn = named.requires(HasName.call(named.c, "Ann")).create();
console.log(IsAnn.rule.predicate === HasName, [...new Queries.Scan(store).select(IsAnn)].map((m: any) => m.c.name));

true [ "Ann" ]


The application holds `HasName` itself, not a copy and not its name: change `HasName` and every rule that applies
it changes with it.

A query can give a predicate's parameters as variables; a validator refuses a predicate with parameters, since it holds
only where it's applied:

In [4]:
console.log([...new Queries.Scan(store).select(HasName, { name: "Bob" })].map((m: any) => m.person.name));
try {
  Validators.Validate(store, [HasName]);
} catch (error) {
  console.log(String(error));
}

[ "Bob" ]


ValueError: predicate 'HasName' has parameters: it is checked where it is applied


## Step 3: written once

A predicate applied in several places is one object, so a snapshot writes it once:

In [5]:
const IsBob = new Predicates.OfPredicate.Builder().name("IsBob").symbols({ c: Contact }).requires(HasName.call(E.variable("c"), "Bob")).create();
const text = SchemaJSON.ToJSON(Constraints.Builders).Reachable(Constraints.OfSet.Schema, Constraints.check([IsAnn, IsBob]));
console.log(text.split('"name": "HasName"').length - 1);

1


## Why it works this way

- **By reference, not by name.** A name is a promise that something with that name exists when the rule is read. A
  reference is the thing itself, written once and read back as one object.
- **Parameters, not copies.** A predicate with parameters is a rule with holes; applying it fills them. That keeps the
  rule in one place.